# Notebook 04: NLP Cleaning and Question Segmentation

This notebook takes OCR output from the previous step, cleans and normalizes the text, and splits it into question-answer pairs for grading.

In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists() and (candidate / "notebooks").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    REPO_ROOT = Path.cwd()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")

In [ ]:
import subprocess
import sys

try:
    import spacy
    from spacy.util import is_package
    if not is_package("en_core_web_sm"):
        subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
except Exception as exc:
    print(f"spaCy setup issue: {exc}")

In [ ]:
from src.nlp_cleaner import NLPCleaner
from src.segmenter import QuestionSegmenter

OCR_JSON = REPO_ROOT / "outputs" / "trocr_results.json"
OUTPUT_JSON = REPO_ROOT / "outputs" / "cleaned_qa.json"
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)

if not OCR_JSON.exists():
    raise FileNotFoundError(f"OCR output not found at {OCR_JSON}. Run the TrOCR notebook first.")

ocr_results = json.loads(OCR_JSON.read_text(encoding="utf-8"))
cleaner = NLPCleaner()
segmenter = QuestionSegmenter()

all_pairs = []
for item in ocr_results:
    raw_text = item.get("text", "")
    normalized = cleaner.normalize_whitespace(raw_text)
    cleaned = cleaner.clean_text(normalized)
    qa_pairs = segmenter.segment_questions_answers(cleaned)

    for qa in qa_pairs:
        all_pairs.append({
            "image": item.get("image", ""),
            "q_no": qa["q_no"],
            "question": qa["question"],
            "answer": qa["answer"],
            "cleaned_answer": cleaner.clean_text(qa["answer"]),
        })

OUTPUT_JSON.write_text(json.dumps(all_pairs, indent=2), encoding="utf-8")
print(f"Saved {len(all_pairs)} cleaned question-answer entries to {OUTPUT_JSON}")

for item in all_pairs[:5]:
    print("-" * 60)
    print(item["q_no"], item["question"][:140])
    print(item["cleaned_answer"][:180])

## Output format

The cleaned output file contains rows with the image name, question number, question text, raw answer, and the cleaned answer string that can be passed to the grading model.